In [1]:
# Setup: DuckDB — zero config, runs in-process, full SQL
# pip install duckdb
import duckdb

# Create an in-memory connection
con = duckdb.connect()

# -------------------------------------------------------
# Seed data: Citi-style server telemetry (employees for ranking demos)
# -------------------------------------------------------

def sql_executer(str):
    con.execute(str)

def sql_executer_printer(str):
    print (con.execute(str).df().to_string(index=False))   

In [2]:
sql="""
CREATE TABLE server_daily_summary (
    server_id TEXT,
    collection_date TEXT,
    avg_cpu REAL,
    region TEXT
);
INSERT INTO server_daily_summary VALUES
    ('srv-01','2026-02-01',72.5,'us-east'),
    ('srv-01','2026-02-02',74.1,'us-east'),
    ('srv-01','2026-02-03',68.9,'us-east'),
    ('srv-01','2026-02-04',79.3,'us-east'),
    ('srv-01','2026-02-05',85.2,'us-east'),
    ('srv-01','2026-02-06',71.0,'us-east'),
    ('srv-01','2026-02-07',88.4,'us-east'),
    ('srv-02','2026-02-01',45.2,'us-east'),
    ('srv-02','2026-02-02',48.1,'us-east'),
    ('srv-02','2026-02-03',51.3,'us-east'),
    ('srv-02','2026-02-04',47.9,'us-east'),
    ('srv-02','2026-02-05',44.0,'us-east'),
    ('srv-02','2026-02-06',52.7,'us-east'),
    ('srv-02','2026-02-07',49.8,'us-east'),
    ('srv-03','2026-02-01',91.3,'us-west'),
    ('srv-03','2026-02-02',88.7,'us-west'),
    ('srv-03','2026-02-03',92.1,'us-west'),
    ('srv-03','2026-02-04',89.5,'us-west'),
    ('srv-03','2026-02-05',95.0,'us-west'),
    ('srv-03','2026-02-06',90.3,'us-west'),
    ('srv-03','2026-02-07',93.8,'us-west'),
    ('srv-04','2026-02-01',33.1,'us-west'),
    ('srv-04','2026-02-02',35.5,'us-west'),
    ('srv-04','2026-02-03',31.8,'us-west'),
    ('srv-04','2026-02-04',38.2,'us-west'),
    ('srv-04','2026-02-05',36.9,'us-west'),
    ('srv-04','2026-02-06',34.4,'us-west'),
    ('srv-04','2026-02-07',37.1,'us-west');
"""
sql_executer(sql)

In [3]:
sql = """
SELECT
    server_id,
    ROUND(AVG(avg_cpu), 2) AS avg_cpu_overall,
    NTILE(4) OVER (ORDER BY AVG(avg_cpu) DESC) AS quartile
FROM server_daily_summary
GROUP BY server_id
ORDER BY avg_cpu_overall DESC
"""
sql_executer_printer(sql)

server_id  avg_cpu_overall  quartile
   srv-03            91.53         1
   srv-01            77.06         2
   srv-02            48.43         3
   srv-04            35.29         4


In [6]:
import pandas as pd
# PERCENT_RANK and CUME_DIST simulation using pandas
# (equivalent to what you would write in production SQL)

df = pd.read_sql_query("""
    SELECT server_id, ROUND(AVG(avg_cpu),2) AS avg_cpu
    FROM server_daily_summary GROUP BY server_id ORDER BY avg_cpu
""", con)

n = len(df)
df = df.sort_values('avg_cpu').reset_index(drop=True)
df['rank'] = df['avg_cpu'].rank(method='min').astype(int)
df['percent_rank'] = ((df['rank'] - 1) / (n - 1)).round(3)
df['cume_dist'] = (df['avg_cpu'].rank(method='max') / n).round(3)

print("PERCENT_RANK and CUME_DIST simulation:")
print(df.to_string(index=False))
print()
print("PERCENT_RANK: (rank-1)/(n-1) — first=0, last=1")
print("CUME_DIST:    rows<=current/n — always > 0")
print()
print("In SQL (requires SQLite 3.44+ or PostgreSQL/Snowflake/DuckDB):")
print("""
SELECT server_id, avg_cpu,
    ROUND(PERCENT_RANK() OVER (ORDER BY avg_cpu) * 100, 1) AS percentile,
    ROUND(CUME_DIST()    OVER (ORDER BY avg_cpu) * 100, 1) AS cumulative_pct
FROM server_daily_summary;
""")

PERCENT_RANK and CUME_DIST simulation:
server_id  avg_cpu  rank  percent_rank  cume_dist
   srv-04    35.29     1         0.000       0.25
   srv-02    48.43     2         0.333       0.50
   srv-01    77.06     3         0.667       0.75
   srv-03    91.53     4         1.000       1.00

PERCENT_RANK: (rank-1)/(n-1) — first=0, last=1
CUME_DIST:    rows<=current/n — always > 0

In SQL (requires SQLite 3.44+ or PostgreSQL/Snowflake/DuckDB):

SELECT server_id, avg_cpu,
    ROUND(PERCENT_RANK() OVER (ORDER BY avg_cpu) * 100, 1) AS percentile,
    ROUND(CUME_DIST()    OVER (ORDER BY avg_cpu) * 100, 1) AS cumulative_pct
FROM server_daily_summary;



C:\Users\shareuser\AppData\Local\Temp\ipykernel_26200\42742950.py:5: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql_query("""
